# Zalo AI Challenge 2026 — LifeMate: Baseline Solution

## 1. Tổng quan Kiến trúc bài toán LifeMate & Cấu hình GitHub Dataset

Cuộc thi **ZAC 2026 - LifeMate** yêu cầu xây dựng trợ lý ảo thông minh (Voice Agent) nhận diện câu lệnh âm thanh tiếng Việt, hiểu ngữ cảnh và gọi chuỗi **Tool Calls** tương ứng trên môi trường mô phỏng [`zaic_tools.ToolEnv`](train+public_test/train/zaic_tools/executor.py).

### Luồng dữ liệu (End-to-End Pipeline):
$$\text{Voice Input (.wav)} \xrightarrow{\text{ASR}} \text{Text} \xrightarrow{\text{Context + Tools Catalog}} \text{Agent Decision} \xrightarrow{\text{ToolEnv}} (\text{tool\_calls}, \text{state\_hash})$$

- **Hỗ trợ Dataset trực tiếp từ GitHub**: Notebook này hỗ trợ tự động clone hoặc stream dữ liệu trực tiếp từ link GitHub Repository của bạn (`https://github.com/vdhwy/ZALOAI.git`), giúp chạy mượt mà trên bất kỳ môi trường nào (máy cá nhân, Google Colab, Kaggle, Docker).

## 2. Cấu hình Dataset từ Link Repo GitHub

In [ ]:
import sys
import os
from pathlib import Path
import json
import torch

# 1. Cấu hình Link GitHub Repo & Thư mục Dữ liệu
GITHUB_REPO_URL = "https://github.com/vdhwy/ZALOAI.git"
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/vdhwy/ZALOAI/main/train+public_test"
DATA_DIR = Path("train+public_test")

# 2. Kiểm tra phần cứng tăng tốc (Apple Silicon MPS / Nvidia CUDA / CPU)
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"PyTorch: {torch.__version__} | Device: {device}")

# 3. Import các module từ baseline.py
from baseline import ensure_dataset, open_dataset, HeuristicBaselineAgent, evaluate_agent, generate_submission

# 4. Đảm bảo dataset có sẵn (tự động clone từ GitHub nếu chưa có local)
data_root = ensure_dataset(repo_url=GITHUB_REPO_URL, data_dir=DATA_DIR)

# 5. Import runtime zaic_tools
from zaic_tools import ToolEnv, schema_validate
print(f"Dataset ready at: {data_root.resolve()}")

## 3. Khám phá Dữ liệu & Test Đọc Trực Tiếp từ GitHub Raw URL

In [ ]:
# Đọc thử mẫu dữ liệu trực tiếp qua đường dẫn GitHub Raw URL
remote_train_url = f"{GITHUB_RAW_BASE}/train/train.jsonl"
print(f"Đang đọc dữ liệu từ: {remote_train_url} ...")

lines = open_dataset(remote_train_url)
first_sample = json.loads(lines[0])

print(f"- Sample ID: {first_sample['id']}")
print(f"- Audio Path: {first_sample['audio']}")
print(f"- Audio Text: {first_sample['audio_text']}")
print(f"- Current Time: {first_sample['context']['current_time']}")
print(f"- Tools được phép gọi: {list(first_sample['context']['installed_apps'].keys())}")

print("\nGround Truth Tool Calls:")
for c in first_sample['label']['tool_calls']:
    print(f" -> {c['name']}({c['arguments']})")

In [ ]:
# Khởi tạo môi trường ToolEnv cho mẫu train_0001
env = ToolEnv("train", first_sample["id"], root=data_root)
print(f"Initial state_hash: {env.get_state_hash()}")

# Chạy thử chuỗi tool call của nhãn
for call in first_sample["label"]["tool_calls"]:
    res = env.call(call["name"], call["arguments"])
    print(f"Executed {call['name']} -> ok: {res.get('ok')}")

print(f"Final state_hash: {env.get_state_hash()}")

## 4. Đánh giá Baseline Agent trên tập Train

Chúng ta đánh giá agent trên tập train để đo **State Hash Accuracy** và **Exact Tool Match**.

In [ ]:
agent = HeuristicBaselineAgent()

# Đánh giá 100 mẫu (có thể dùng remote_train_url hoặc local file)
train_source = f"{GITHUB_RAW_BASE}/train/train.jsonl"
results = evaluate_agent(agent, data_source=train_source, data_root=data_root, split="train", max_samples=100)
print(f"\n>>> State Hash Accuracy: {results['accuracy']*100:.2f}%")

## 5. Sinh File Nộp Bài Public Test (`submission.jsonl`) & Kiểm Tra Tính Hợp Lệ

In [ ]:
# Tạo file kết quả submission.jsonl cho 634 mẫu public test
test_source = f"{GITHUB_RAW_BASE}/public_test/public_test.jsonl"
output_file = Path("submission.jsonl")

generate_submission(agent, test_source=test_source, output_path=output_file, data_root=data_root, split="public_test")

# Hiển thị 3 dòng đầu tiên
with open(output_file, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().strip())